# Rag Arena Experiments

This notebook demonstrates the multi-dataset RAG workflow used by Rag Arena.

In [ ]:
import json
import os
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

api_keys_path = project_root / 'api_keys.json'
if api_keys_path.exists():
    api_keys = json.loads(api_keys_path.read_text(encoding='utf-8'))
    if api_keys.get('OPENROUTER_API_KEY'):
        os.environ['OPENROUTER_API_KEY'] = api_keys['OPENROUTER_API_KEY']

from rag_arena.data import load_qa_split
from rag_arena.experiments import ExperimentConfig, run_experiment
from rag_arena.indexing import build_corpus
from rag_arena.retrieval import build_retriever

In [ ]:
samples = load_qa_split(dataset_name='hotpotqa', dataset_config='distractor', split='validation', sample_size=5, seed=42)
documents = build_corpus(samples)
retriever = build_retriever(documents, {'method': 'bm25', 'top_k': 5})
samples[0].question, retriever.retrieve(samples[0].question, top_k=3)[0].metadata

In [ ]:
import time

date_time_format = "%Y-%m-%d_%H:%M:%S"
now = time.time()

# Using hotpotqa and ollama
hotpot_config = ExperimentConfig(
    dataset_name='hotpotqa',
    dataset_config='distractor',
    sample_size=200,
    output_dir=f'outputs/hotpotqa_iterative_{time.strftime(date_time_format, time.localtime(now))}',
    retriever_config={
        'method': 'iterative',
        'base_method': 'hybrid',
        'top_k': 50,
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'alpha': 0.5,
        'max_iter': 3,
        'sim_threshold': 0.85,
    },
    rerank_config={
        'enabled': True,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'top_k': 10,
    },
    generation_config={
        'provider': 'ollama',
        'model_name': 'qwen3.5:2b',
        'temperature': 0.0,
        'max_tokens': 8192,
    },
)
hotpot_result = run_experiment(hotpot_config)
hotpot_result.summary

In [ ]:
# Using 2Wiki

date_time_format = "%Y-%m-%d_%H:%M:%S"
now = time.time()

wiki2_config = ExperimentConfig(
    dataset_name='2wikimultihopqa',
    dataset_config='validation',
    sample_size=200,
    output_dir=f'outputs/2wikimultihopqa_iterative_{time.strftime(date_time_format, time.localtime(now))}',
    retriever_config={
        'method': 'iterative',
        'base_method': 'hybrid',
        'top_k': 50,
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'alpha': 0.5,
        'max_iter': 3,
        'sim_threshold': 0.85,
    },
    rerank_config={
        'enabled': True,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'top_k': 10,
    },
    generation_config={
        'provider': 'ollama',
        'model_name': 'qwen3.5:2b',
        'temperature': 0.0,
        'max_tokens': 8192,
    },
)
wiki2_result = run_experiment(wiki2_config)
wiki2_result.summary

In [ ]:
# Using 2Wiki and openrouter

wiki2_config = ExperimentConfig(
    dataset_name='2wikimultihopqa',
    dataset_split='validation',
    sample_size=5,
    output_dir='outputs/notebook_2wiki_smoke',
    retriever_config={
        'method': 'iterative',
        'base_method': 'hybrid',
        'top_k': 10,
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'alpha': 0.5,
        'max_iter': 3,
        'sim_threshold': 0.85,
    },
    rerank_config={
        'enabled': True,
        'model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'top_k': 5,
    },
    generation_config={
        'provider': 'openrouter',
        # 'model_name': 'qwen/qwen3.6-plus-preview:free',
        'model_name': 'nvidia/nemotron-3-super-120b-a12b:free',
        'temperature': 0.0,
        'max_tokens': 8192,
    },
)
wiki2_result = run_experiment(wiki2_config)
wiki2_result.summary